In [1]:
!pip install fastf1

In [2]:
import os
import json
import time
import requests
import pandas as pd
import fastf1
from fastf1 import RateLimitExceededError

try:
    from fastf1.api import SessionNotAvailableError
except ImportError:
    from fastf1._api import SessionNotAvailableError

/home/emili-tabuti/Documentos/projects/tcc-f1/.venv/lib/python3.12/site-packages/fastf1/api.py:32: UserWarning: `fastf1.api` will be considered private in future releases and potentially be removed or changed!
  warnings.warn("`fastf1.api` will be considered private in future releases and "


In [3]:
# URL Jolpica
BASE_URL = "https://api.jolpi.ca/ergast/f1"

# roda so localmente (VS Code): o Colab foi abandonado porque a fonte oficial
# do fastf1 (livetiming.formula1.com) bloqueia o runtime do Colab com 403 e o
# espelho (livetiming-mirror.fastf1.dev) devolve 404 pras mesmas sessoes,
# tanto pra temporadas antigas quanto recentes
PASTA_DADOS = "dados"

# pasta usada pelo FastF1 para armazenar cache
CACHE_DIR = os.path.join(PASTA_DADOS, "cache")

# arquivo que guarda quais rounds do fastf1 já foram baixados com sucesso
CKPT_FILE = os.path.join(PASTA_DADOS, "fastf1_checkpoint.json")

# pasta onde vamos salvar a base consolidada (etapa de juncao, mais pra frente)
PASTA_SAIDA = os.path.join(PASTA_DADOS, "processados")

# cria as pastas caso elas ainda não existam
os.makedirs(PASTA_DADOS, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(PASTA_SAIDA, exist_ok=True)

# ativa o cache do FastF1
fastf1.Cache.enable_cache(CACHE_DIR)

In [4]:
def requisitar(url, tentativas=5):
    """faz um GET na url, tentando de novo se der erro de conexão ou limite de requisições (429)"""

    for _ in range(tentativas):

        try:
            response = requests.get(url, timeout=30)

        except requests.exceptions.RequestException as erro:
            print("Erro de conexão:", erro)
            time.sleep(10)
            continue

        # código 429 significa excesso de requisições
        if response.status_code == 429:
            print("Limite da API. Aguardando...")
            time.sleep(30)
            continue

        if response.status_code != 200:
            print("Erro:", response.status_code)
            return None

        return response.json()

    return None


def paginar(url, limit=100):
    """percorre todas as páginas de um endpoint paginado da jolpica, uma de cada vez"""

    offset = 0

    while True:
        separador = "&" if "?" in url else "?"
        data = requisitar(f"{url}{separador}limit={limit}&offset={offset}")

        if data is None:
            return

        yield data

        total = int(data["MRData"]["total"])
        offset += limit
        time.sleep(0.3)

        if offset >= total:
            return


def arquivo_existe(nome_arquivo):
    """verifica se um arquivo ja foi extraido/salvo na pasta de dados, pra nao refazer a busca na api"""
    return os.path.exists(os.path.join(PASTA_DADOS, nome_arquivo))


def salvar(df, nome_arquivo):
    """salva o dataframe como csv dentro da pasta de dados"""
    caminho = os.path.join(PASTA_DADOS, nome_arquivo)
    df.to_csv(caminho, index=False)
    print("Salvo:", caminho)


def salvar_incremental(df, nome_arquivo):
    """acrescenta linhas num csv, criando o cabeçalho só se o arquivo ainda não existir"""
    caminho = os.path.join(PASTA_DADOS, nome_arquivo)
    existe = os.path.exists(caminho)
    df.to_csv(caminho, mode="a", header=not existe, index=False)


def carregar_checkpoint():
    """le quais rounds do fastf1 já foram processados em execuções anteriores"""
    if os.path.exists(CKPT_FILE):
        with open(CKPT_FILE) as f:
            return set(json.load(f))
    return set()


def salvar_checkpoint(concluidos):
    with open(CKPT_FILE, "w") as f:
        json.dump(sorted(concluidos), f)

In [5]:
# RESULTADOS DAS CORRIDAS DE 2018 ATÉ 2025
def buscar_resultados():

    # ja extraido antes, nao busca de novo na api
    if arquivo_existe("resultados_2018_2025.csv"):
        print("resultados_2018_2025.csv já existe, pulando.")
        return

    print("\nBuscando resultados de 2018 até 2025...")

    dados = []

    for ano in range(2018, 2026):
        print("Temporada:", ano)

        for pagina in paginar(f"{BASE_URL}/{ano}/results.json"):
            for race in pagina["MRData"]["RaceTable"]["Races"]:
                for result in race["Results"]:
                    dados.append({
                        "season": race["season"],
                        "round": race["round"],
                        "race_name": race["raceName"],
                        "driver_id": result["Driver"]["driverId"],
                        "constructor_id": result["Constructor"]["constructorId"],
                        "grid_position": result["grid"],
                        "finish_position": result["position"],
                        "status": result["status"],
                        "points": result["points"],
                        "laps": result.get("laps", "")
                    })

    salvar(pd.DataFrame(dados), "resultados_2018_2025.csv")

In [6]:
# PIT STOPS 2018 ATÉ 2025
def buscar_pitstops():

    # ja extraido antes, nao busca de novo na api
    if arquivo_existe("pitstops_2018_2025.csv"):
        print("pitstops_2018_2025.csv já existe, pulando.")
        return

    print("\nBuscando pit stops de 2018 até 2025...")

    dados = []

    for ano in range(2018, 2026):
        print("Temporada:", ano)

        # busca o calendário da temporada pra saber quantos rounds ela teve
        calendario = requisitar(f"{BASE_URL}/{ano}/races.json?limit=100")
        if not calendario:
            continue

        rounds = [r["round"] for r in calendario["MRData"]["RaceTable"]["Races"]]

        for round_num in rounds:
            for pagina in paginar(f"{BASE_URL}/{ano}/{round_num}/pitstops.json"):
                for race in pagina["MRData"]["RaceTable"]["Races"]:
                    for pit in race["PitStops"]:
                        dados.append({
                            "season": ano,
                            "round": round_num,
                            "race_name": race["raceName"],
                            "driver_id": pit["driverId"],
                            "stop": pit["stop"],
                            "lap": pit["lap"],
                            "duration": pit["duration"]
                        })

            time.sleep(0.5)

    salvar(pd.DataFrame(dados), "pitstops_2018_2025.csv")

In [7]:
# CIRCUITOS UTILIZADOS ENTRE 2018 E 2025
def buscar_circuitos():

    # ja extraido antes, nao busca de novo na api
    if arquivo_existe("circuitos_2018_2025.csv"):
        print("circuitos_2018_2025.csv já existe, pulando.")
        return

    print("\nBuscando circuitos de 2018 até 2025...")

    dados = []

    for ano in range(2018, 2026):
        print("Temporada:", ano)

        data = requisitar(f"{BASE_URL}/{ano}/circuits.json?limit=100")
        if not data:
            continue

        for circuito in data["MRData"]["CircuitTable"]["Circuits"]:
            dados.append({
                "circuit_id": circuito["circuitId"],
                "circuit_name": circuito["circuitName"],
                "lat": circuito["Location"]["lat"],
                "long": circuito["Location"]["long"],
                "country": circuito["Location"]["country"]
            })

        time.sleep(0.5)

    # um circuito pode aparecer em várias temporadas, então remove as repetições pelo id
    df = pd.DataFrame(dados).drop_duplicates(subset="circuit_id")
    salvar(df, "circuitos_2018_2025.csv")

In [8]:
# PILOTOS QUE COMPETIRAM ENTRE 2018 E 2025
def buscar_pilotos():

    # ja extraido antes, nao busca de novo na api
    if arquivo_existe("pilotos_2018_2025.csv"):
        print("pilotos_2018_2025.csv já existe, pulando.")
        return

    print("\nBuscando pilotos de 2018 até 2025...")

    dados = []

    for ano in range(2018, 2026):
        print("Temporada:", ano)

        data = requisitar(f"{BASE_URL}/{ano}/drivers.json?limit=100")
        if not data:
            continue

        for driver in data["MRData"]["DriverTable"]["Drivers"]:
            dados.append({
                "driver_id": driver["driverId"],
                # codigo oficial de 3 letras (ex: "HAM"), usado pra casar com o fastf1
                "code": driver.get("code", ""),
                "given_name": driver["givenName"],
                "family_name": driver["familyName"],
                "date_of_birth": driver.get("dateOfBirth", ""),
                "nationality": driver.get("nationality", "")
            })

        time.sleep(0.5)

    df = pd.DataFrame(dados).drop_duplicates(subset="driver_id")
    salvar(df, "pilotos_2018_2025.csv")

In [9]:
# CALENDARIO: EM QUAL CIRCUITO CADA CORRIDA ACONTECEU
def buscar_calendario_circuitos():

    # ja extraido antes, nao busca de novo na api
    if arquivo_existe("calendario_circuitos_2018_2025.csv"):
        print("calendario_circuitos_2018_2025.csv já existe, pulando.")
        return

    print("\nBuscando calendário de circuitos de 2018 até 2025...")

    dados = []

    for ano in range(2018, 2026):
        print("Temporada:", ano)

        data = requisitar(f"{BASE_URL}/{ano}/races.json?limit=100")
        if not data:
            continue

        for race in data["MRData"]["RaceTable"]["Races"]:
            dados.append({
                "season": race["season"],
                "round": race["round"],
                "race_name": race["raceName"],
                "circuit_id": race["Circuit"]["circuitId"]
            })

        time.sleep(0.5)

    salvar(pd.DataFrame(dados), "calendario_circuitos_2018_2025.csv")

In [10]:
# FASTF1 - QUALIFYING, VOLTAS E CLIMA DE 2018 ATÉ 2025
def cols_disponiveis(df, colunas):
    """devolve só as colunas da lista que realmente existem no dataframe"""
    return [coluna for coluna in colunas if coluna in df.columns]


def carregar_sessao(
    ano, round_num, tipo,
    tentativas_curtas=4, espera_curta=30,
    tentativas_longas=3, espera_longa=3700,
    tentativas_indisponivel=2, espera_indisponivel=15,
):
    """carrega uma sessão do fastf1, tentando de novo se bater no limite da api ou se
    a sessão vier indisponível/vazia no espelho de dados do fastf1.

    o RateLimitExceededError vem de um limitador client-side do próprio fastf1 pra
    chamadas na api jolpica/ergast (usada, entre outras coisas, pelo fallback que
    busca o tempo da 1ª volta de cada piloto quando a fonte principal não traz esse
    dado): no máximo 200 chamadas numa janela deslizante de 60 minutos (ver
    `fastf1/req.py`). como a janela é deslizante (não um bloqueio fixo), depois que
    ela libera de novo as próximas chamadas tendem a liberar logo em seguida (elas
    costumam ter sido feitas em rajada, próximas no tempo) - por isso, ao bater no
    limite, primeiro tenta algumas vezes com espera curta, e só recorre à espera
    longa (~1h, o tamanho real da janela) se as tentativas curtas não resolverem.

    já a sessão indisponível costuma ser permanente pra corridas antigas (o espelho
    simplesmente não tem os dados arquivados) - por isso só tenta mais uma vez,
    rápido, pra cobrir uma eventual instabilidade passageira, sem desperdiçar tempo
    tentando de novo algo que não vai se resolver.

    se os orçamentos de retry se esgotarem, relança o erro original pra quem chamou
    saber que esse round não foi concluído (e assim não marcar o checkpoint como feito).
    """
    tentativas_indisponivel_restantes = tentativas_indisponivel

    for tentativa_longa in range(tentativas_longas):
        tentativas_curtas_restantes = tentativas_curtas

        while True:
            try:
                sessao = fastf1.get_session(ano, round_num, tipo)
                sessao.load(laps=True, telemetry=False, weather=(tipo == "R"), messages=False)

                # o load() pode "passar" sem erro mesmo quando os dados essenciais
                # não vieram (o fastf1 só loga um aviso internamente nesses casos)
                if tipo == "R" and sessao.laps.empty:
                    raise SessionNotAvailableError("laps veio vazio após o load")
                if tipo == "Q" and sessao.results.empty:
                    raise SessionNotAvailableError("results veio vazio após o load")

                return sessao

            except RateLimitExceededError:
                if tentativas_curtas_restantes <= 0:
                    break  # esgotou as tentativas curtas, escala pra espera longa
                tentativas_curtas_restantes -= 1
                print(f"Limite da API atingido, aguardando {espera_curta}s antes de tentar de novo...")
                time.sleep(espera_curta)

            except SessionNotAvailableError as erro:
                if tentativas_indisponivel_restantes <= 0:
                    raise
                tentativas_indisponivel_restantes -= 1
                print(f"Sessão indisponível no espelho do fastf1 ({erro}), aguardando {espera_indisponivel}s antes de tentar de novo...")
                time.sleep(espera_indisponivel)

        if tentativa_longa < tentativas_longas - 1:
            print(f"Limite da API continua ativo, aguardando {espera_longa}s (janela de ~1h) antes de tentar de novo...")
            time.sleep(espera_longa)

    raise RateLimitExceededError("limite da api continua ativo após várias tentativas")


def buscar_fastf1():

    # se os 3 arquivos ja existem, assume que a extracao ja foi concluida antes
    arquivos_finais = [
        "fastf1_qualifying_2018_2025.csv",
        "fastf1_laps_2018_2025.csv",
        "fastf1_weather_2018_2025.csv",
    ]
    if all(arquivo_existe(nome) for nome in arquivos_finais):
        print("Arquivos do FastF1 já existem, pulando.")
        return

    print("\nBuscando dados do FastF1...")

    # rounds que já foram baixados com sucesso em execuções anteriores
    concluidos = carregar_checkpoint()

    for ano in range(2018, 2026):
        print("\nFastF1 - Temporada:", ano)

        try:
            schedule = fastf1.get_event_schedule(ano, include_testing=False)
        except Exception as erro:
            print("Erro ao buscar calendário:", erro)
            continue

        rounds = schedule["RoundNumber"].dropna().astype(int)

        for round_num in rounds:
            chave = f"{ano}-{round_num}"

            # pula rounds que já foram baixados numa execução anterior
            if chave in concluidos:
                print("Round:", round_num, "(já processado, pulando)")
                continue

            print("Round:", round_num)

            # só marca o round como concluído se qualifying e corrida forem
            # extraídos com sucesso; qualquer erro (limite da api, sessão
            # indisponível, falha de rede etc.) deixa o round pendente pra
            # ser retomado na próxima execução, em vez de pular ele pra sempre
            round_ok = True

            # QUALIFYING
            try:
                session_q = carregar_sessao(ano, round_num, "Q")
                resultados = session_q.results

                colunas = cols_disponiveis(resultados, ["Abbreviation", "Position", "Q1", "Q2", "Q3"])
                df_q = resultados[colunas].copy()
                df_q = df_q.rename(columns={"Abbreviation": "Driver", "Position": "position"})

                df_q["season"] = ano
                df_q["round"] = round_num

                salvar_incremental(df_q, "fastf1_qualifying_2018_2025.csv")

            except Exception as erro:
                print("Erro no qualifying:", ano, round_num, erro)
                round_ok = False

            # CORRIDA / VOLTAS
            try:
                session_r = carregar_sessao(ano, round_num, "R")

                colunas_laps = cols_disponiveis(session_r.laps, [
                    "Driver", "LapNumber", "LapTime", "Sector1Time", "Sector2Time",
                    "Sector3Time", "Compound", "TyreLife", "Stint", "TrackStatus",
                    "FreshTyre", "PitInTime", "PitOutTime"
                ])

                df_laps = session_r.laps[colunas_laps].copy()
                df_laps["season"] = ano
                df_laps["round"] = round_num
                salvar_incremental(df_laps, "fastf1_laps_2018_2025.csv")

                # CLIMA
                if session_r.weather_data is not None and len(session_r.weather_data) > 0:
                    colunas_weather = cols_disponiveis(
                        session_r.weather_data,
                        ["AirTemp", "Humidity", "Rainfall", "TrackTemp", "WindSpeed"]
                    )

                    df_weather = session_r.weather_data[colunas_weather].copy()
                    df_weather["season"] = ano
                    df_weather["round"] = round_num
                    salvar_incremental(df_weather, "fastf1_weather_2018_2025.csv")

            except Exception as erro:
                print("Erro na corrida:", ano, round_num, erro)
                round_ok = False

            # só marca como concluído se não houve nenhuma falha no round
            if round_ok:
                concluidos.add(chave)
                salvar_checkpoint(concluidos)

            time.sleep(1)

    print("\nFastF1 finalizado (ou pausado pelo limite da api - rode de novo mais tarde pra continuar de onde parou).")

In [11]:
buscar_resultados()
buscar_pitstops()
buscar_circuitos()
buscar_pilotos()
buscar_calendario_circuitos()
buscar_fastf1()

print("\nExtração finalizada.")
print("Arquivos salvos em:", PASTA_DADOS)

resultados_2018_2025.csv já existe, pulando.
pitstops_2018_2025.csv já existe, pulando.
circuitos_2018_2025.csv já existe, pulando.
pilotos_2018_2025.csv já existe, pulando.
calendario_circuitos_2018_2025.csv já existe, pulando.
Arquivos do FastF1 já existem, pulando.

Extração finalizada.
Arquivos salvos em: dados


## Etapa 2 — Consolidação da base

Junta todos os csv extraídos acima numa única tabela, com uma linha por (temporada, corrida, piloto). Essa é a mesma lógica do `juntar_base.py` do repositório, adaptada pra rodar direto no notebook.

In [12]:
# dados de curadoria manual (altitude, numero de curvas, comprimento, tipo de
# circuito) que nao vem de nenhuma api publica. embutido aqui pra o notebook
# nao depender de subir esse arquivo manualmente.
CIRCUITOS_MANUAL = [
    ("albert_park", "Albert Park Circuit", 13, 16, 5.278, 0),
    ("bahrain", "Bahrain International Circuit", 7, 15, 5.412, 0),
    ("jeddah", "Jeddah Corniche Circuit", 15, 27, 6.174, 1),
    ("shanghai", "Shanghai International Circuit", 5, 16, 5.451, 0),
    ("miami", "Miami International Autodrome", 2, 19, 5.412, 1),
    ("imola", "Autodromo Enzo e Dino Ferrari", 26, 19, 4.909, 0),
    ("monaco", "Circuit de Monaco", 7, 19, 3.337, 1),
    ("catalunya", "Circuit de Barcelona-Catalunya", 109, 16, 4.675, 0),
    ("villeneuve", "Circuit Gilles Villeneuve", 7, 14, 4.361, 1),
    ("red_bull_ring", "Red Bull Ring", 678, 10, 4.318, 0),
    ("silverstone", "Silverstone Circuit", 145, 18, 5.891, 0),
    ("hungaroring", "Hungaroring", 264, 14, 4.381, 0),
    ("spa", "Circuit de Spa-Francorchamps", 401, 19, 7.004, 0),
    ("zandvoort", "Circuit Zandvoort", 3, 14, 4.259, 0),
    ("monza", "Autodromo Nazionale di Monza", 162, 11, 5.793, 0),
    ("baku", "Baku City Circuit", 0, 20, 6.003, 1),
    ("marina_bay", "Marina Bay Street Circuit", 0, 19, 4.940, 1),
    ("suzuka", "Suzuka Circuit", 44, 18, 5.807, 0),
    ("losail", "Losail International Circuit", 20, 16, 5.380, 0),
    ("americas", "Circuit of the Americas", 150, 20, 5.513, 0),
    ("rodriguez", "Autodromo Hermanos Rodriguez", 2285, 17, 4.304, 0),
    ("interlagos", "Autodromo Jose Carlos Pace", 785, 15, 4.309, 0),
    ("vegas", "Las Vegas Strip Circuit", 620, 17, 6.201, 1),
    ("yas_marina", "Yas Marina Circuit", 3, 16, 5.281, 0),
    ("mugello", "Autodromo Internazionale del Mugello", 379, 15, 5.245, 0),
    ("portimao", "Autodromo Internacional do Algarve", 108, 15, 4.653, 0),
    ("nurburgring", "Nurburgring", 322, 16, 5.148, 0),
    ("istanbul", "Istanbul Park", 130, 14, 5.338, 0),
    ("bahrain_outer", "Bahrain International Circuit (Outer)", 7, 11, 3.543, 0),
    ("sochi", "Sochi Autodrom", 10, 18, 5.848, 1),
    ("hockenheimring", "Hockenheimring", 110, 16, 4.574, 0),
    ("ricard", "Circuit Paul Ricard", 406, 15, 5.842, 0),
]

# se ja existe (ex: alguem editou o csv na mao), nao sobrescreve
if arquivo_existe("circuitos_manual.csv"):
    print("circuitos_manual.csv já existe, pulando.")
else:
    df_circuitos_manual = pd.DataFrame(
        CIRCUITOS_MANUAL,
        columns=["circuit_id", "circuit_name", "altitude_m", "corners", "length_km", "circuit_type"]
    )
    df_circuitos_manual.to_csv(os.path.join(PASTA_DADOS, "circuitos_manual.csv"), index=False)
    print("circuitos_manual.csv gravado em", PASTA_DADOS)

circuitos_manual.csv já existe, pulando.


In [13]:
# a api marca o Sakhir GP (2020, layout "Outer Circuit") com o mesmo
# circuit_id do Bahrain GP normal ("bahrain"), mas o circuitos_manual trata
# como um circuito a parte ("bahrain_outer") porque o layout da pista foi
# diferente. corrige na mao so esse caso especial.
CORRIGIR_CIRCUITO_POR_CORRIDA = {
    "Sakhir Grand Prix": "bahrain_outer",
}


def carregar_csv(nome_arquivo):
    """le um csv de dentro da pasta de dados brutos"""
    caminho = os.path.join(PASTA_DADOS, nome_arquivo)
    return pd.read_csv(caminho)


def tempo_para_segundos(coluna):
    """converte uma coluna de tempo (formato "0 days 00:01:32.123") para segundos"""
    return pd.to_timedelta(coluna, errors="coerce").dt.total_seconds()


def duracao_pitstop_para_segundos(valor):
    """converte a duracao do pitstop pra segundos"""
    valor = str(valor)

    if ":" in valor:
        minutos, segundos = valor.split(":")
        return int(minutos) * 60 + float(segundos)

    try:
        return float(valor)
    except ValueError:
        return None

In [14]:
# RESULTADOS + CIRCUITO
def montar_resultados_com_circuito():
    """pega os resultados das corridas e descobre o circuito de cada uma

    o circuito vem do calendario_circuitos_2018_2025.csv, que traz o
    circuit_id oficial de cada (season, round) direto da api.
    """

    print("Montando resultados com circuito...")

    resultados = carregar_csv("resultados_2018_2025.csv")
    calendario = carregar_csv("calendario_circuitos_2018_2025.csv")

    calendario = calendario[["season", "round", "circuit_id"]]
    resultados = resultados.merge(calendario, on=["season", "round"], how="left")

    # corrige o caso especial do Sakhir GP (CORRIGIR_CIRCUITO_POR_CORRIDA)
    for race_name, circuito_correto in CORRIGIR_CIRCUITO_POR_CORRIDA.items():
        resultados.loc[resultados["race_name"] == race_name, "circuit_id"] = circuito_correto

    sem_circuito = resultados[resultados["circuit_id"].isna()]
    if len(sem_circuito) > 0:
        print("Aviso: corridas sem circuito mapeado ->", sem_circuito["race_name"].unique())

    return resultados

In [15]:
# CIRCUITOS (localizacao da api + dados manuais de altitude, curvas,...)
def montar_circuitos():
    """junta a localizacao (vinda da api) com os dados manuais de cada circuito"""

    print("Montando tabela de circuitos...")

    circuitos_api = carregar_csv("circuitos_2018_2025.csv")
    circuitos_manual = carregar_csv("circuitos_manual.csv")

    localizacao = circuitos_api[["circuit_id", "lat", "long", "country"]]
    circuitos = circuitos_manual.merge(localizacao, on="circuit_id", how="left")

    return circuitos

In [16]:
# PIT STOPS (agregado por corrida e piloto)
def montar_pitstops_agregado():
    """conta quantas paradas cada piloto fez numa corrida e o tempo gasto nelas"""

    print("Agregando pit stops...")

    pitstops = carregar_csv("pitstops_2018_2025.csv")
    pitstops["duration"] = pitstops["duration"].apply(duracao_pitstop_para_segundos)

    agregado = pitstops.groupby(["season", "round", "driver_id"]).agg(
        num_pitstops=("stop", "count"),
        tempo_total_pitstop=("duration", "sum"),
        tempo_medio_pitstop=("duration", "mean"),
    ).reset_index()

    return agregado

In [17]:
# VOLTAS DO FASTF1 (agregado por corrida e piloto)
def montar_laps_agregado(codigo_para_driver_id):
    """resume as voltas de cada piloto numa corrida: tempo medio, melhor volta,..."""

    print("Agregando voltas (fastf1)...")

    laps = carregar_csv("fastf1_laps_2018_2025.csv")

    # troca o codigo de 3 letras do fastf1 pelo driver_id usado no resto da base
    laps["driver_id"] = laps["Driver"].map(codigo_para_driver_id)
    laps["LapTime_s"] = tempo_para_segundos(laps["LapTime"])

    agregado = laps.groupby(["season", "round", "driver_id"]).agg(
        fastf1_avg_lap_time=("LapTime_s", "mean"),
        fastf1_best_lap_time=("LapTime_s", "min"),
        fastf1_num_voltas=("LapNumber", "count"),
        fastf1_num_stints=("Stint", "nunique"),
        fastf1_tyre_life_media=("TyreLife", "mean"),
    ).reset_index()

    composto_mais_usado = (
        laps.groupby(["season", "round", "driver_id"])["Compound"]
        .agg(lambda serie: serie.mode().iloc[0] if not serie.mode().empty else "UNKNOWN")
        .reset_index()
        .rename(columns={"Compound": "tire_compound_predominante"})
    )

    agregado = agregado.merge(composto_mais_usado, on=["season", "round", "driver_id"], how="left")

    return agregado

In [18]:
# QUALIFYING (fastf1)
def montar_qualifying(codigo_para_driver_id):
    """pega a posicao de largada e os tempos de classificacao de cada piloto"""

    print("Montando qualifying (fastf1)...")

    quali = carregar_csv("fastf1_qualifying_2018_2025.csv")

    quali["driver_id"] = quali["Driver"].map(codigo_para_driver_id)
    quali["Q1_s"] = tempo_para_segundos(quali["Q1"])
    quali["Q2_s"] = tempo_para_segundos(quali["Q2"])
    quali["Q3_s"] = tempo_para_segundos(quali["Q3"])
    quali = quali.rename(columns={"position": "qualifying_position"})

    colunas = ["season", "round", "driver_id", "qualifying_position", "Q1_s", "Q2_s", "Q3_s"]
    return quali[colunas]

In [19]:
# CLIMA (fastf1)
def montar_weather_agregado():
    """resume o clima da corrida inteira (uma linha por temporada+round)"""

    print("Agregando clima (fastf1)...")

    weather = carregar_csv("fastf1_weather_2018_2025.csv")

    agregado = weather.groupby(["season", "round"]).agg(
        temp_ar_media=("AirTemp", "mean"),
        temp_pista_media=("TrackTemp", "mean"),
        umidade_media=("Humidity", "mean"),
        vento_media=("WindSpeed", "mean"),
        choveu=("Rainfall", "max"),  # se choveu em algum momento, marca a corrida como chuvosa
    ).reset_index()

    return agregado

In [20]:
caminho_saida = os.path.join(PASTA_SAIDA, "base_consolidada_2018_2025.csv")

# se a base ja foi consolidada antes, nao refaz a juncao
if os.path.exists(caminho_saida):
    print("Base consolidada já existe, pulando junção:", caminho_saida)
    print("Se quiser refazer, apague esse arquivo antes de rodar de novo.")
else:
    pilotos = carregar_csv("pilotos_2018_2025.csv")

    # monta o mapa codigo-de-3-letras -> driver_id a partir da propria coluna "code"
    codigo_para_driver_id = (
        pilotos.dropna(subset=["code"])
        .set_index("code")["driver_id"]
        .to_dict()
    )

    resultados = montar_resultados_com_circuito()
    circuitos = montar_circuitos()
    pitstops_agg = montar_pitstops_agregado()
    laps_agg = montar_laps_agregado(codigo_para_driver_id)
    quali = montar_qualifying(codigo_para_driver_id)
    weather_agg = montar_weather_agregado()

    print("Juntando tudo numa base só...")

    base = resultados.merge(circuitos, on="circuit_id", how="left")
    base = base.merge(pilotos, on="driver_id", how="left")
    base = base.merge(pitstops_agg, on=["season", "round", "driver_id"], how="left")
    base = base.merge(laps_agg, on=["season", "round", "driver_id"], how="left")
    base = base.merge(quali, on=["season", "round", "driver_id"], how="left")
    base = base.merge(weather_agg, on=["season", "round"], how="left")

    base.to_csv(caminho_saida, index=False)

    print("Base consolidada salva em:", caminho_saida)
    print("Linhas:", len(base), "| Colunas:", len(base.columns))

Base consolidada já existe, pulando junção: dados/processados/base_consolidada_2018_2025.csv
Se quiser refazer, apague esse arquivo antes de rodar de novo.


## Etapa 3 — Tratamento de DNF

Classifica cada linha da base consolidada em: terminou a corrida, abandonou por culpa própria (bateu, rodou, saiu da pista) ou abandonou por falha mecânica / motivo não identificado. Só quem terminou ou abandonou por culpa própria entra no treino do modelo — abandono mecânico é um evento aleatório, sem relação com o desempenho do piloto naquele dia. Mesma lógica do `tratar_dnf.py` do repositório, adaptada pra rodar direto no notebook.

In [ ]:
# quem terminou a corrida (ou terminou com voltas de atraso, mas foi classificado)
STATUS_CLASSIFICADO = {"finished", "lapped"}

# abandono por culpa do piloto (bateu, rodou, saiu da pista,...)
PALAVRAS_DNF_PILOTO = [
    "accident",
    "collision",
    "spun off",
    "spin",
    "crash",
    "damage",
]


def normalizar_status(status):
    """deixa o texto do status em minusculo e sem espacos nas pontas"""
    return str(status).strip().lower()


def classificado(status_norm):
    """verifica se o piloto terminou a corrida"""
    if status_norm in STATUS_CLASSIFICADO:
        return True

    # ex: "+1 Lap", "+2 Laps"
    if status_norm.startswith("+") and "lap" in status_norm:
        return True

    return False


def contem_alguma_palavra(status_norm, lista_palavras):
    """verifica se o status contem alguma das palavras da lista"""
    return any(palavra in status_norm for palavra in lista_palavras)


def classificar_categoria(status):
    """decide a categoria da linha: classificado, dnf_piloto ou dnf_mecanico"""

    status_norm = normalizar_status(status)

    if classificado(status_norm):
        return "classificado"

    if contem_alguma_palavra(status_norm, PALAVRAS_DNF_PILOTO):
        return "dnf_piloto"

    # dnf_mecanico: falha mecanica (motor, cambio, freio,...) e os casos
    # que nao ficou claro (ex: "Retired", "Disqualified", "Did not start", "Withdrew")
    return "dnf_mecanico"

In [22]:
ARQUIVO_SAIDA_TREINO = os.path.join(PASTA_SAIDA, "base_dnf_treino_2018_2025.csv")
ARQUIVO_SAIDA_EXCLUIDOS = os.path.join(PASTA_SAIDA, "base_dnf_excluidos_2018_2025.csv")

print("Lendo base consolidada...")
base = pd.read_csv(caminho_saida)

print("Classificando cada linha (terminou / dnf piloto / dnf mecanico)...")
base["status_categoria"] = base["status"].apply(classificar_categoria)

print()
print(base["status_categoria"].value_counts())

# so entra no treino quem terminou a corrida ou abandonou por culpa do piloto
# abandono mecanico/motivo desconhecido fica de fora do treino do modelo
entra_no_treino = base["status_categoria"].isin(["classificado", "dnf_piloto"])

base_treino = base[entra_no_treino].copy()
base_excluidos = base[~entra_no_treino].copy()

base_treino.to_csv(ARQUIVO_SAIDA_TREINO, index=False)
base_excluidos.to_csv(ARQUIVO_SAIDA_EXCLUIDOS, index=False)

print()
print("Linhas que entram no treino:", len(base_treino))
print("Linhas excluidas (dnf mecanico/motivo desconhecido):", len(base_excluidos))
print()
print("Base de treino salva em:", ARQUIVO_SAIDA_TREINO)
print("Base excluida salva em:", ARQUIVO_SAIDA_EXCLUIDOS)

Lendo base consolidada...
Classificando cada linha (terminou / dnf piloto / dnf mecanico)...

status_categoria
classificado    2943
dnf_mecanico     368
dnf_piloto       147
Name: count, dtype: int64

Linhas que entram no treino: 3090
Linhas excluidas (dnf mecanico/motivo desconhecido): 368

Base de treino salva em: dados/processados/base_dnf_treino_2018_2025.csv
Base excluida salva em: dados/processados/base_dnf_excluidos_2018_2025.csv
